<a href="https://colab.research.google.com/github/Aniruddha-Ray/Atlas-Guided-Medical-Image-Segmentation/blob/main/Atlas_Guided_Medical_Image_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Data Preparation

## UNETR Model

In [ ]:
import torch
from monai.networks.nets import UNETR
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

In [ ]:
model = UNETR(
    in_channels=1,
    out_channels=3,
    img_size=(96, 96, 96),
    feature_size=16,
    hidden_size=768,
    mlp_dim=3072,
    num_heads=12,
    proj_type="perceptron",   # newer MONAI versions
    norm_name="instance",
    res_block=True,
).to(device)

In [ ]:
checkpoint = torch.load(
    "pretrained_unetr.pth",
    map_location=device
)

In [ ]:
model.load_state_dict(
    checkpoint["state_dict"],
    strict=False
)

In [ ]:
criterion = DiceCELoss(
    to_onehot_y=True,
    softmax=True
)

In [ ]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad,
           model.parameters()),
    lr=1e-4,
    weight_decay=1e-5
)

In [ ]:
dice_metric = DiceMetric(
    include_background=False,
    reduction="mean"
)

In [ ]:
num_epochs = 100

for epoch in range(num_epochs):

    model.train()

    epoch_loss = 0.0

    for images, masks in train_loader:

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        # Forward
        outputs = model(images)

        # Compute loss
        loss = criterion(outputs, masks)

        # Backprop
        loss.backward()

        # Update weights
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Loss: {avg_loss:.4f}"
    )


    # Validation

    model.eval()

    with torch.no_grad():

        dice_metric.reset()

        for val_images, val_masks in val_loader:

            val_images = val_images.to(device)
            val_masks = val_masks.to(device)

            preds = model(val_images)

            dice_metric(y_pred=preds,
                        y=val_masks)

        mean_dice = dice_metric.aggregate().item()

    print(
        f"Validation Dice: {mean_dice:.4f}"
    )



In [ ]:
torch.save(
    model.state_dict(),
    "unetr_finetuned.pth"
)

## UNet Model

In [ ]:
import torch
import torch.nn as nn
from monai.networks.nets import UNet

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

In [ ]:
backbone = UNet(
    spatial_dims=2,
    in_channels=3,
    out_channels=16,  # dummy
    channels=(32, 64, 128, 256),
    strides=(2, 2, 2),
).to(device)

In [ ]:
class UNetClassifier(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()

        self.backbone = backbone

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Linear(
            16,
            num_classes
        )

    def forward(self, x):

        x = self.backbone(x)

        x = self.pool(x)

        x = torch.flatten(x, 1)

        x = self.classifier(x)

        return x


model = UNetClassifier(
    backbone,
    num_classes=5
).to(device)

In [ ]:
criterion = DiceCELoss(
    to_onehot_y=True,
    softmax=True
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

In [ ]:

num_epochs = 50

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()

        total += labels.size(0)

    train_acc = correct / total

    print(
        f"Epoch {epoch+1} | "
        f"Loss={running_loss/len(train_loader):.4f} | "
        f"Train Acc={train_acc:.4f}"
    )

    # Validation

    model.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            preds = outputs.argmax(dim=1)

            val_correct += (
                preds == labels
            ).sum().item()

            val_total += labels.size(0)

    val_acc = val_correct / val_total

    print(
        f"Validation Acc={val_acc:.4f}"
    )



In [ ]:
torch.save(
    {
        "state_dict": model.state_dict()
    },
    "unet_classifier.pth"
)

## CNN + FC

In [ ]:
import torch
import torch.nn as nn

class SimpleCNN(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.AdaptiveAvgPool2d(1),

            nn.Flatten(),

            nn.Linear(
                128,
                256
            ),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                256,
                num_classes
            )
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

In [ ]:
model = SimpleCNN(
    num_classes=5
).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
outputs.shape = (B, num_classes)

labels.shape = (B,)

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [ ]:
num_epochs = 20

for epoch in range(num_epochs):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    train_acc = correct / total

    print(
        f"Epoch {epoch+1} "
        f"Loss={running_loss/len(train_loader):.4f} "
        f"Acc={train_acc:.4f}"
    )

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        predictions = outputs.argmax(dim=1)

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

val_acc = correct / total

print(
    f"Validation Accuracy: {val_acc:.4f}"
)

In [ ]:
torch.save(
    {
        "state_dict": model.state_dict()
    },
    "cnn_classifier.pth"
)

## Simple ViT

In [ ]:
import torch
import torch.nn as nn
from transformers import ViTModel, ViTConfig

In [ ]:
class ViTClassifier(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        self.vit = ViTModel.from_pretrained(
            "google/vit-base-patch16-224"
        )

        self.classifier = nn.Linear(
            768,
            num_classes
        )

    def forward(self, x):

        outputs = self.vit(x)

        cls_token = outputs.last_hidden_state[:, 0]

        logits = self.classifier(cls_token)

        return logits

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

In [ ]:
model = ViTClassifier(
    num_classes=5
).to(device)

In [ ]:
for param in model.parameters():
    param.requires_grad = True

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

In [ ]:
num_epochs = 20

for epoch in range(num_epochs):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        preds = outputs.argmax(dim=1)

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

    train_acc = correct / total

    print(
        f"Epoch {epoch+1} "
        f"Loss={running_loss/len(train_loader):.4f} "
        f"Acc={train_acc:.4f}"
    )

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        preds = outputs.argmax(dim=1)

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

val_acc = correct / total

print(
    f"Validation Accuracy: {val_acc:.4f}"
)

In [ ]:
torch.save(
    {
        "state_dict": model.state_dict()
    },
    "vit_classifier.pth"
)

In [ ]:
model.load_state_dict(
    torch.load(
        "vit_classifier.pth"
    )["state_dict"]
)

model.eval()

In [ ]:
with torch.no_grad():

    logits = model(image)

    pred_class = logits.argmax(dim=1)

print(pred_class.item())++